In [ ]:
!nvidia-smi

In [ ]:
!find /content -name "best.pt" 2>/dev/null

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip -q install kaggle

import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

assert "kaggle.json" in uploaded, (
    "Önce kaggle.json dosyasını yüklemelisin."
)

with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])

os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download \
    -d umuttuygurr/aerial-uav-thermal-inferred-unified-dataset \
    -p /content/dataset_zip

In [ ]:
import os
import glob
import zipfile

os.makedirs("/content/dataset_zip", exist_ok=True)
os.makedirs("/content/dataset", exist_ok=True)

zip_files = glob.glob("/content/dataset_zip/*.zip")

if not zip_files:
    raise FileNotFoundError(
        "ZIP bulunamadı. Önce Adım 3'teki indirme hücresini çalıştır."
    )

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/dataset")

print("Veri seti açıldı.")

In [ ]:
from pathlib import Path

root = Path("/content/dataset")

for p in list(root.rglob("*"))[:100]:
    print(p.relative_to(root))

In [ ]:
from pathlib import Path

root = Path("/content/dataset")

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

images = [
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
]

labels = list(root.rglob("*.txt"))
yamls = list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))

print("Görüntü sayısı:", len(images))
print("TXT dosyası sayısı:", len(labels))
print("YAML dosyaları:")
for p in yamls:
    print(p)

print("\nÖrnek görüntüler:")
for p in images[:10]:
    print(p)

print("\nÖrnek TXT dosyaları:")
for p in labels[:10]:
    print(p)

In [ ]:
from pathlib import Path
from collections import Counter

root = Path("/content/dataset")
labels = list(root.rglob("*.txt"))

class_counts = Counter()
bad_lines = []
empty_files = 0

for label_file in labels:
    try:
        lines = label_file.read_text().splitlines()
    except Exception as e:
        bad_lines.append((str(label_file), str(e)))
        continue

    if not lines:
        empty_files += 1

    for line_no, line in enumerate(lines, start=1):
        parts = line.split()

        if len(parts) != 5:
            bad_lines.append(
                (str(label_file), line_no, "5 alan bekleniyor", line)
            )
            continue

        try:
            cls = int(parts[0])
            coords = [float(x) for x in parts[1:]]
        except ValueError:
            bad_lines.append(
                (str(label_file), line_no, "Sayısal olmayan değer", line)
            )
            continue

        if cls < 0:
            bad_lines.append(
                (str(label_file), line_no, "Negatif sınıf numarası", line)
            )
            continue

        if not all(0 <= x <= 1 for x in coords):
            bad_lines.append(
                (str(label_file), line_no, "Koordinatlar 0-1 dışında", line)
            )
            continue

        if coords[2] <= 0 or coords[3] <= 0:
            bad_lines.append(
                (str(label_file), line_no, "Genişlik/yükseklik sıfır veya negatif", line)
            )
            continue

        class_counts[cls] += 1

print("Geçerli etiket sınıf dağılımı:", dict(sorted(class_counts.items())))
print("Boş etiket dosyası:", empty_files)
print("Hatalı kayıt sayısı:", len(bad_lines))

for error in bad_lines[:20]:
    print(error)

In [ ]:
from pathlib import Path

root = Path("/content/dataset")
yamls = list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))

for y in yamls:
    print(f"\n--- {y} ---")
    print(y.read_text()[:10000])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/UAV_Thermal_Project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Proje klasörü:", PROJECT_DIR)

In [ ]:
!pip -q install ultralytics

In [ ]:
DATA_YAML = "/content/dataset/merged_uav_dataset/data.yaml"

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path("/content/dataset/merged_uav_dataset")
YAML_PATH = DATASET_DIR / "data.yaml"

print("Dataset klasörü mevcut:", DATASET_DIR.exists())
print("YAML mevcut:", YAML_PATH.exists())

if YAML_PATH.exists():
    print("\n--- data.yaml içeriği ---")
    print(YAML_PATH.read_text())

print("\n--- Dataset klasörleri ---")
for p in DATASET_DIR.iterdir():
    print(p.name, "->", "klasör" if p.is_dir() else "dosya")

print("\n--- images altındaki klasörler ---")
images_dir = DATASET_DIR / "images"

if images_dir.exists():
    for p in images_dir.rglob("*"):
        if p.is_dir():
            print(p)

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path("/content/dataset/merged_uav_dataset")
YAML_PATH = DATASET_DIR / "data.yaml"

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_DIR.resolve())

for split in ["train", "val", "test"]:
    split_dir = DATASET_DIR / "images" / split

    if split_dir.exists():
        data[split] = f"images/{split}"
        print(f"{split}: {split_dir} bulundu")
    elif split == "test":
        data.pop("test", None)
        print("Test klasörü yok; test alanı kaldırıldı.")
    else:
        raise FileNotFoundError(
            f"{split_dir} bulunamadı. Klasör yapısını kontrol et."
        )

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print("\nDüzeltilmiş data.yaml:")
print(YAML_PATH.read_text())

In [ ]:
from google.colab import drive
import os


drive.mount('/content/drive')


os.makedirs('/content/drive/MyDrive/yolov8_runs', exist_ok=True)

from ultralytics import YOLO


model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=25,
    imgsz=640,
    batch=32,
    patience=10,
    device=0,
    workers=4,
    cache='disk',
    project='/content/drive/MyDrive/yolov8_runs',
    name='uav_thermal_fast',
    exist_ok=True,
    seed=42,
    plots=True
)

In [ ]:
from pathlib import Path
import shutil

run_dir = Path("/content/yolov8_runs/uav_thermal")
weights_dir = run_dir / "weights"

save_dir = Path(
    "/content/drive/MyDrive/UAV_Thermal_Project/YOLOv8"
)
save_dir.mkdir(parents=True, exist_ok=True)

for filename in ["best.pt", "last.pt"]:
    src = weights_dir / filename
    if src.exists():
        shutil.copy2(src, save_dir / filename)

for filename in ["results.csv", "results.png", "confusion_matrix.png"]:
    src = run_dir / filename
    if src.exists():
        shutil.copy2(src, save_dir / filename)

print("Mevcut eğitim çıktıları Drive'a kopyalandı.")